# 05 — Asynchronous AI Pipelines
## Overlapping I/O without overwhelming the service you call


**Rule:** every instructional example is complete and executable. Only the final project is intentionally unfinished.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you. The async cells use top-level `await`, which Jupyter supports directly.

You will see:

- **Predict** — before you run a cell, write down what you expect it to print, and why.
- **What you just saw** — a short note after a cell.
- **`assert` lines** — the specification.

In [1]:
from __future__ import annotations

import random
from dataclasses import dataclass
from typing import Any, Callable, Sequence

SEED = 7
random.seed(SEED)
print(f"Ready. Seed = {SEED}")

Ready. Seed = 7


## Before we start

Async is worth it when your program spends most of its time **waiting** — for a model API, a database, a file. While one task waits, the event loop lets another task run.

Keep these five concerns separate; the project combines them at the end:

- **Concurrency limit** — how many operations may be running at once.
- **Rate limit** — how fast operations may be *started* (per second).
- **Timeout** — how long one operation may take before you give up on it.
- **Retry policy** — which failures you try again, and how gently.
- **Fault isolation** — how one failed item is reported without losing the others.

Async helps I/O-bound work. It does **not** make CPU-heavy Python run in parallel — that still blocks the loop.

## 1. Concurrency is not parallelism

Coroutines take turns on **one** thread. They hand control back to the loop at each `await`. So four tasks that each `await asyncio.sleep(0.05)` overlap their waiting and finish in about `0.05 s` total — not `0.20 s`.

**Predict.** `concurrency_demo()` starts 4 requests, each sleeping `0.05 s`, all with `asyncio.gather`. About how long does it take?

![Async timeline](assets/async_timeline.svg)

In [2]:
import asyncio, time

async def simulated_request(name: str, delay: float) -> str:
    await asyncio.sleep(delay)          # stand-in for a network call
    return f"{name} done"

async def concurrency_demo() -> tuple[list[str], float]:
    start = time.perf_counter()
    results = await asyncio.gather(*(simulated_request(f"job-{i}", 0.05) for i in range(4)))
    return results, time.perf_counter() - start

results, elapsed = await concurrency_demo()
print("results:", results)
print(f"elapsed: {elapsed*1000:.0f} ms   <- about 50 ms, not 200 ms: the four waits overlapped")

assert elapsed < 0.15

results: ['job-0 done', 'job-1 done', 'job-2 done', 'job-3 done']
elapsed: 51 ms   <- about 50 ms, not 200 ms: the four waits overlapped


### What you just saw

`asyncio.gather` starts all four coroutines, then waits for all of them. They were mostly asleep at the same time, so total time ≈ the slowest one.

This only helps *waiting* work. A coroutine that runs a long CPU loop with no `await` blocks every other task on the loop.

## 2. Cap how many run at once: a semaphore

A **semaphore** with limit `n` lets at most `n` tasks into a block at a time; the rest wait their turn. Use it to protect a connection pool or memory. Wrap it around the smallest expensive part.

Note: it limits *how many at once*, not *how fast they start* — that is the next section.

In [3]:
async def bounded_map(items: Sequence[int], limit: int) -> list[int]:
    if limit <= 0:
        raise ValueError("concurrency limit must be positive")
    semaphore = asyncio.Semaphore(limit)
    async def worker(item: int) -> int:
        async with semaphore:              # at most `limit` workers inside here at once
            await asyncio.sleep(0.01)
            return item * item
    return await asyncio.gather(*(worker(i) for i in items))

out = await bounded_map([1, 2, 3, 4], limit=2)
print("bounded_map([1,2,3,4], limit=2):", out, " <- results stay in input order")
assert out == [1, 4, 9, 16]

bounded_map([1,2,3,4], limit=2): [1, 4, 9, 16]  <- results stay in input order


In [4]:
try:
    await bounded_map([1, 2], limit=0)
except ValueError as exc:
    print("limit=0 refused:", exc)
else:
    raise AssertionError("Expected an invalid semaphore limit")

limit=0 refused: concurrency limit must be positive


### What you just saw

Results come back in **input** order because `gather` preserves the order of the coroutines you pass it — not the order they finish in.

`asyncio.Semaphore(0)` is a perfectly valid object that just never lets anyone in, so `bounded_map` checks `limit` itself rather than relying on the primitive to complain.

## 3. Cap how fast they start: a token bucket

A **token bucket** holds up to `capacity` tokens and refills at `rate` tokens per second. Starting a request costs one token. So:

- `capacity` sets how big an initial **burst** you allow.
- `rate` sets the long-run average.

A lock protects the token count. The sleep happens *outside* the lock, so a waiting caller does not block others from updating the count.

**Predict.** `TokenBucket(rate=100, capacity=2)`, then 3 `acquire()` calls at once. The first two are instant (the burst). Roughly how long until the third?

In [5]:
class TokenBucket:
    def __init__(self, rate: float, capacity: float):
        if rate <= 0 or capacity <= 0:
            raise ValueError("rate and capacity must be positive")
        self.rate, self.capacity, self.tokens = rate, capacity, capacity
        self.updated = time.monotonic()          # monotonic: never jumps backward
        self.lock = asyncio.Lock()

    async def acquire(self) -> None:
        while True:
            async with self.lock:
                now = time.monotonic()
                self.tokens = min(self.capacity, self.tokens + (now - self.updated) * self.rate)
                self.updated = now
                if self.tokens >= 1:
                    self.tokens -= 1
                    return
                delay = (1 - self.tokens) / self.rate
            await asyncio.sleep(delay)            # wait OUTSIDE the lock

bucket = TokenBucket(rate=100, capacity=2)
start = time.perf_counter()
await asyncio.gather(*(bucket.acquire() for _ in range(3)))
print(f"3 acquires took {(time.perf_counter()-start)*1000:.0f} ms")
print("2 were free (the burst); the 3rd waited ~10 ms for the bucket to refill 1 token at rate 100/s")

3 acquires took 10 ms
2 were free (the burst); the 3rd waited ~10 ms for the bucket to refill 1 token at rate 100/s


### What you just saw

A token = permission to start one request. You start full (`capacity` tokens), so the first `capacity` calls go immediately. After that, elapsed time refills tokens at `rate` per second, and a caller waits until one whole token is available.

This controls the *start rate*. It does not limit how many run at once — for that you still need a semaphore.

## 4. Timeouts, retries, and "is it safe to repeat?"

- A **timeout** bounds how long you wait for one call.
- A **retry** should only apply to failures that might be temporary (a timeout, a dropped connection) — not to a validation error or a permission error, which will fail again.
- **Backoff** means waiting longer between attempts. **Jitter** means adding a little randomness so a thousand clients do not all retry at the same instant.
- Only retry an operation that is **safe to repeat**. If it has a side effect (charging a card, sending a message), a retry can do it twice — use an idempotency key or do not auto-retry.

**Predict.** `flaky` fails once with `ConnectionError`, then succeeds. How many times does `retry(flaky)` call it?

In [6]:
async def retry(operation: Callable[[], Any], attempts: int = 3, base_delay: float = 0.01):
    if attempts <= 0:
        raise ValueError("attempts must be positive")
    if base_delay < 0:
        raise ValueError("base_delay must be non-negative")

    last_error = None
    for attempt in range(attempts):
        try:
            return await asyncio.wait_for(operation(), timeout=0.2)
        except (TimeoutError, ConnectionError) as error:      # only these are "maybe temporary"
            last_error = error
            if attempt + 1 < attempts:
                await asyncio.sleep(base_delay * 2**attempt + random.random() * base_delay)
    raise last_error

calls = 0
async def flaky():
    global calls
    calls += 1
    if calls < 2:
        raise ConnectionError("temporary")
    return "ok"

result = await retry(flaky)
print("retry(flaky) ->", result)
print("flaky was called", calls, "times  (1 failure + 1 success)")
assert result == "ok" and calls == 2

retry(flaky) -> ok
flaky was called 2 times  (1 failure + 1 success)


In [7]:
# Bad configuration is refused.
for a, d in ((0, 0.01), (-1, 0.01), (1, -0.01)):
    try:
        await retry(flaky, attempts=a, base_delay=d)
    except ValueError as exc:
        print(f"attempts={a}, base_delay={d} -> {exc}")
    else:
        raise AssertionError("bad retry config should raise")

# A non-temporary error is NOT retried -- it propagates on the first try.
async def permanent_failure():
    raise ValueError("invalid request")

try:
    await retry(permanent_failure)
except ValueError as exc:
    print("permanent_failure ->", exc, " (raised immediately, not retried)")
else:
    raise AssertionError("ValueError should not be retried")

attempts=0, base_delay=0.01 -> attempts must be positive
attempts=-1, base_delay=0.01 -> attempts must be positive
attempts=1, base_delay=-0.01 -> base_delay must be non-negative
permanent_failure -> invalid request  (raised immediately, not retried)


## 5. One failure should not sink the batch

`asyncio.gather(..., return_exceptions=True)` stops one failure from cancelling the others — but then your results list is a mix of values and raw exception objects, which is awkward.

Better: give every item a small typed **envelope** with either a value or an error string. Success and failure become plain data.

In [8]:
@dataclass(frozen=True)
class Outcome:
    item: int
    value: str | None = None
    error: str | None = None

async def safe_worker(item: int) -> Outcome:
    try:
        if item == 2:
            raise RuntimeError("simulated failure")
        await asyncio.sleep(0.005)
        return Outcome(item, value=f"answer-{item}")
    except Exception as exc:
        return Outcome(item, error=f"{type(exc).__name__}: {exc}")

outcomes = await asyncio.gather(*(safe_worker(i) for i in range(4)))
for o in outcomes:
    print(o)
print("failures:", sum(o.error is not None for o in outcomes), " successes:", sum(o.value is not None for o in outcomes))

assert sum(o.error is not None for o in outcomes) == 1

Outcome(item=0, value='answer-0', error=None)
Outcome(item=1, value='answer-1', error=None)
Outcome(item=2, value=None, error='RuntimeError: simulated failure')
Outcome(item=3, value='answer-3', error=None)
failures: 1  successes: 3


### What you just saw

You get one `Outcome` per input, in order. Item 2 failed; items 0, 1, 3 still have their results, in their original positions. One collection to inspect, no surprises.

Do not dump raw exception text (or payloads) into an envelope or a log by default — pick a safe error category, and attach a correlation ID when an operator needs to dig into one request.

## Project — Resilient batch inference client

Build a client with bounded concurrency, token-bucket rate limiting, per-call timeouts, retry with jitter, structured outcomes and cancellation cleanup.

**Suggested test matrix:**

- No more than the configured number of operations are in flight.
- Input order is preserved even when completion order differs.
- The token bucket permits its configured initial burst and limits the long-run start rate.
- Timeouts become structured failures and do not leave tasks running.
- Only transient failures are retried; permanent failures stop immediately.
- Retry configuration rejects zero attempts and negative delays.
- Outcomes include one result per input, including failures.
- Cancellation propagates cleanly and leaves no pending tasks.
- Repeated operations use idempotency keys where side effects are possible.
- Latency, retry count and failure categories are measurable without leaking payloads.

**Acceptance criteria:** never exceed the configured concurrency; preserve input order; retry only transient failures; expose latency metrics; leave no pending tasks after cancellation.

You may `from course_utils import TokenBucket, retry, Outcome, bounded_map` instead of copying the cells above; the module ships the same implementations with docstrings.

**Checks to run yourself**

- Track a live counter inside the worker and assert its max never exceeds the concurrency limit.
- Give the client a batch where item 3 always times out; assert its `Outcome` is a structured failure and the others succeed.
- Cancel `run_batch` mid-flight, then assert `asyncio.all_tasks()` holds nothing of yours.
- Submit the same idempotency key twice concurrently and assert the side effect happens once.
- Time a batch of N slow calls under `rate=r` and check the elapsed time is at least `N/r` minus the burst.

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Reuse asyncio.Semaphore plus TokenBucket, retry and the Outcome envelope from this
# notebook, or import them:
#     from course_utils import TokenBucket, retry, Outcome
#
# 1. ResilientInferenceClient(concurrency, rate, capacity, timeout, attempts):
#      validate every argument.
# 2. run_batch(items): bounded concurrency + token-bucket pacing + per-call timeout
#    + retry-with-jitter for transient errors only. Return one Outcome per input,
#    in input order.
# 3. Cancellation: on cancel, leave no task pending.
# 4. Metrics: latency, retry count, failure category -- without logging payloads.

class ResilientInferenceClient:
    ...


raise NotImplementedError("Implement the resilient batch inference client")
